# Safety Rails and Sandboxing
Companion notebook for the **Safety Rails and Sandboxing** section of Chapter 8.

Put hard controls in code, not only in prompts -- prompts can be ignored or changed by bad input, but a validation function in your codebase can't be talked out of its rules.

This is the book's `safe_read_file` example. `read_file` is a stand-in so the function is actually callable:

In [1]:
def read_file(path: str) -> str:
    """Stand-in for a real filesystem read."""
    return f"<contents of {path}>"


def safe_read_file(path: str):
    if ".." in path or not path.startswith("/reports/") or not path.endswith(".csv"):
        raise ValueError("path not allowed")
    return read_file(path)

In [2]:
print(safe_read_file("/reports/q3_summary.csv"))

for bad_path in ["/reports/../../secrets.env", "/etc/passwd", "/reports/notes.txt"]:
    try:
        safe_read_file(bad_path)
    except ValueError as e:
        print(f"{bad_path!r} rejected: {e}")

<contents of /reports/q3_summary.csv>
'/reports/../../secrets.env' rejected: path not allowed
'/etc/passwd' rejected: path not allowed
'/reports/notes.txt' rejected: path not allowed


**Allow lists**: expose only approved tools and approved arguments -- the model shouldn't be able to invoke anything just by describing it in text.

In [3]:
ALLOWED_TOOLS = {"safe_read_file", "get_customer_tier"}


def call_tool(tool_name: str, **kwargs):
    if tool_name not in ALLOWED_TOOLS:
        raise PermissionError(f"tool not on the allow list: {tool_name}")
    return globals()[tool_name](**kwargs)


print(call_tool("safe_read_file", path="/reports/q3_summary.csv"))
try:
    call_tool("run_shell_command", command="rm -rf /")
except PermissionError as e:
    print(f"blocked: {e}")

<contents of /reports/q3_summary.csv>
blocked: tool not on the allow list: run_shell_command


For genuinely risky actions (code execution, shell access, file writes), a **sandbox** isolates the run -- limited filesystem access, restricted network, no secrets in the runtime. A minimal illustration of the *shape* of that boundary (not a real sandbox, which needs OS-level isolation, not just a Python wrapper):

In [4]:
def run_in_sandbox(fn, *, allow_network=False, allow_filesystem=False):
    """Illustrates the boundary a real sandbox enforces at the OS level --
    this wrapper only documents the intent, it does not itself isolate anything."""
    print(f"[sandbox] network={allow_network} filesystem={allow_filesystem} -- running {fn.__name__}")
    return fn()


run_in_sandbox(lambda: print("computing a pure result with no I/O"), allow_network=False, allow_filesystem=False)

[sandbox] network=False filesystem=False -- running <lambda>
computing a pure result with no I/O
